In [70]:
%run 00_config.py

In [71]:
from datasets import load_dataset
import os
import json

##### Run get_product_metadata to get product-train.jsonl 

In [80]:
am_products = []
with open('product-train.jsonl') as f:
    for line in f:
        am_products.append(json.loads(line))

In [ ]:
#Sample product JSON for 5 rows
import random
random_elements = random.sample(am_products, 5)
for product in random_elements:
    print(f"ID: {product.get('Uniq Id')}\n\t\t Name: {product.get('Product Name')}\n\t\t Desc: {product.get('About Product')}\n\t\t Images:{product.get('Image')}")
        

##### For an Image url like this: `https://abcd/efg/hij/51%2B1w0CK0ML.jpg`

- the output file name will be *{productid}*\_51_2B1w0CK0ML\_*{imageNumForProduct}*.jpg
- the imageNumForProduct means it's the Nth image for that product and it typically only goes up to 7
- Some images might be missing from the URL ie. a 404
- Matplot and pyplot can't read % signs in image file names

In [87]:
import urllib.request as ur
from urllib.error import HTTPError
def download_image(image_url, image_file_name, product_id, image_index):
    try:
        [image_file_name,image_extension] = image_file_name.split('.') # assume only one dot in the file name 
    except ValueError as e:
        print(f"ValueError: {e} for image {image_url}")
        return
    destination = f"{dl_images_folder}/{product_id}_{image_file_name}_{image_index+1}.{image_extension}".replace('%','_')
    if(not (os.path.exists(f"{destination}"))): # if file does not exist, then download
        #print(f"Downloading image: {image_url} to {destination}") # verbose printing for each file
        try:
            resource = ur.urlretrieve(image_url, destination)
        except HTTPError as e:
            print(f"HTTP Error: {e.code} for image {image_url}")
        except Exception as e:
            print(f"Generic Non-HTTP Error: {e.code} for image {image_url}")

##### We ignore transparent-pixel files. They are not good data

In [ ]:
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
count = 0
image_list = []
for product in am_products:
    unique_id = product.get('Uniq Id')
    product_image_list = product.get('Image').split("|")
    for i, image_url in enumerate(product_image_list):
        if "transparent-pixel" in image_url.lower():
            continue
        image_file_name = os.path.basename(image_url)
        image_list.append((image_url, image_file_name, unique_id, i))
        #image_list tuples will be metadata to name the downloadedfiles


#### After this code cell runs, expect 33,975 downloaded files. If you have fewer than that, re-run the cell and check and re-run again

- Some files will throw HTTP and Value Errors because of bad data
- You can adjust the thread count yourself. Default is 8

In [ ]:
with ThreadPoolExecutor(max_workers=8) as executor:
    results = list(executor.map(lambda image_meta: download_image(*image_meta), image_list))
